# NB-02: QKV Projections and Multi-Head Attention Layout

## Learning Objectives
- Understand how Q, K, V linear projections work in Wan2.1 — all three are dim-to-dim (no expansion), with head splitting done post-projection via einops
- Compare the two multi-head layout conventions: (B, S, N, D) for flash_attn_2/3 vs (B, N, S, D) for PyTorch SDPA and SageAttn
- Verify that splitting into heads and merging back is a lossless reshape operation — no information is lost

## Prerequisites
- **Prior notebooks:** NB-01 (RMSNorm is used to normalize q/k before RoPE in SelfAttention)
- **Assumed concepts:** Linear layers (nn.Linear), tensor reshape/permute operations, multi-head attention concept (splitting dim into N heads of size D)

## Concept Map
- QKV projections → used in SelfAttention.forward (NB-04) and CrossAttention.forward (NB-04)
- (B,S,N,D) layout → used by flash_attention when flash_attn_2/3 backend is active (NB-04)
- (B,N,S,D) layout → used by flash_attention when PyTorch SDPA fallback is active (NB-04, the CPU path)
- head_dim = dim // num_heads → determines RoPE frequency band sizes (NB-03)

In [ ]:
import sys
import types
import importlib.util
import pathlib

# Find project root: walk up from Course/ to find the directory containing diffsynth/
# This handles both normal checkout and git worktree scenarios (up to 6 levels up).
_here = pathlib.Path().resolve()  # where notebook is executed from
PROJECT_ROOT = None
for _candidate in [_here] + list(_here.parents)[:6]:
    if (_candidate / "diffsynth" / "models" / "wan_video_dit.py").exists():
        PROJECT_ROOT = _candidate
        break
if PROJECT_ROOT is None:
    raise FileNotFoundError(
        "Could not find diffsynth/models/wan_video_dit.py. "
        "Run this notebook from Course/ inside the project checkout."
    )
print(f"Project root: {PROJECT_ROOT}")

# Stub wan_video_camera_controller (not needed for DiT primitive demos)
_cam_stub = types.ModuleType("diffsynth.models.wan_video_camera_controller")
_cam_stub.SimpleAdapter = type("SimpleAdapter", (), {"__init__": lambda s, *a, **kw: None})
_diffsynth_stub = types.ModuleType("diffsynth")
_models_stub = types.ModuleType("diffsynth.models")
sys.modules["diffsynth"] = _diffsynth_stub
sys.modules["diffsynth.models"] = _models_stub
sys.modules["diffsynth.models.wan_video_camera_controller"] = _cam_stub

# Load wan_video_dit.py directly, bypassing the broken diffsynth/__init__.py chain
# Source: importlib.util.spec_from_file_location
_dit_path = PROJECT_ROOT / "diffsynth" / "models" / "wan_video_dit.py"
_spec = importlib.util.spec_from_file_location("diffsynth.models.wan_video_dit", _dit_path)
dit = importlib.util.module_from_spec(_spec)
sys.modules["diffsynth.models.wan_video_dit"] = dit
_spec.loader.exec_module(dit)

# Import symbols used in this notebook
from diffsynth.models.wan_video_dit import SelfAttention, RMSNorm

import torch
import torch.nn as nn
from einops import rearrange

print("Setup complete.")

## 1. QKV Projections

In Wan2.1's `SelfAttention` module, queries (Q), keys (K), and values (V) are each produced by a separate `nn.Linear(dim, dim)` projection — three independent linear layers with **no dimension expansion**. The output shape is identical to the input shape: `[B, S, dim]`.

This is a deliberate design choice: the head splitting is done **after** the projection, using einops. Contrast this with some implementations that project directly to `[dim // num_heads]` per head — Wan2.1 uses full-dim projections and then reshapes.

Source: `diffsynth/models/wan_video_dit.py`, line 125

In [ ]:
# Source: diffsynth/models/wan_video_dit.py, line 132-135
B, S, dim, num_heads = 1, 20, 1536, 12
head_dim = dim // num_heads            # 128

x = torch.randn(B, S, dim)            # [B, S, dim]
q_proj = nn.Linear(dim, dim)           # dim -> dim (no expansion)
k_proj = nn.Linear(dim, dim)           # dim -> dim (no expansion)
v_proj = nn.Linear(dim, dim)           # dim -> dim (no expansion)

q = q_proj(x)                          # [B, S, dim] -> [B, S, dim]
k = k_proj(x)                          # [B, S, dim] -> [B, S, dim]
v = v_proj(x)                          # [B, S, dim] -> [B, S, dim]

assert q.shape == torch.Size([B, S, dim])
assert k.shape == torch.Size([B, S, dim])
assert v.shape == torch.Size([B, S, dim])
print(f"Q shape: {q.shape}  (dim={dim} = {num_heads} heads x {head_dim} head_dim)")
print(f"K shape: {k.shape}")
print(f"V shape: {v.shape}")
print(f"head_dim = {dim} // {num_heads} = {head_dim}")
print(f"Each projection has {dim * dim + dim:,} parameters (weight: {dim}x{dim}, bias: {dim})")

## 2. Multi-Head Splitting — Two Layout Conventions

After the QKV projections, the flat `dim`-sized vectors are split into `num_heads` heads, each of size `head_dim = dim // num_heads`. This is done via a single `einops.rearrange` call — no data is copied, just the shape metadata changes.

**Two competing conventions exist in the codebase**, driven by different attention backends:

| Convention | Shape | Backend | Axis order |
|------------|-------|---------|------------|
| Sequence-first | `(B, S, N, D)` | flash_attn_2, flash_attn_3 | batch, sequence, heads, head_dim |
| Head-first | `(B, N, S, D)` | PyTorch SDPA, SageAttn | batch, heads, sequence, head_dim |

The `flash_attention` function in Wan2.1 handles both — it selects the rearrange pattern based on which backend is available at runtime. On CPU (no flash_attn installed), it falls back to PyTorch SDPA with the `(B, N, S, D)` layout.

Source: `diffsynth/models/wan_video_dit.py`, line 28

In [ ]:
# Source: diffsynth/models/wan_video_dit.py, lines 30-60 (flash_attention)

# Convention 1: flash_attn_2/3 — sequence-first layout
q_bsnd = rearrange(q, "b s (n d) -> b s n d", n=num_heads)  # [B, S, N, D]
k_bsnd = rearrange(k, "b s (n d) -> b s n d", n=num_heads)  # [B, S, N, D]
v_bsnd = rearrange(v, "b s (n d) -> b s n d", n=num_heads)  # [B, S, N, D]

# Convention 2: PyTorch SDPA / SageAttn — head-first layout
q_bnsd = rearrange(q, "b s (n d) -> b n s d", n=num_heads)  # [B, N, S, D]
k_bnsd = rearrange(k, "b s (n d) -> b n s d", n=num_heads)  # [B, N, S, D]
v_bnsd = rearrange(v, "b s (n d) -> b n s d", n=num_heads)  # [B, N, S, D]

assert q_bsnd.shape == torch.Size([B, S, num_heads, head_dim])
assert q_bnsd.shape == torch.Size([B, num_heads, S, head_dim])
print(f"(B,S,N,D) layout: {q_bsnd.shape}  — used by flash_attn_2/3")
print(f"(B,N,S,D) layout: {q_bnsd.shape}  — used by PyTorch SDPA (CPU fallback)")
print(f"In both cases: N={num_heads} heads, D={head_dim} head_dim")

## 3. Round-Trip Verification — Head Splitting Is Lossless

Because head splitting is a pure reshape (no computation, no data movement), merging the heads back must produce the exact original tensor. This is an important property: the attention mechanism operates on the split representation, but the output is merged back before the final output projection `self.o`.

Verifying this round-trip guarantees that einops `rearrange` is being used correctly — any off-by-one in the head count or dimension ordering would produce a mismatch.

In [ ]:
# Merge heads back: (B,N,S,D) -> (B,S,N*D) = (B,S,dim)
q_restored = rearrange(q_bnsd, "b n s d -> b s (n d)")      # [B, S, dim]
assert q_restored.shape == torch.Size([B, S, dim]), \
    f"Expected {(B, S, dim)}, got {q_restored.shape}"
assert torch.allclose(q_restored, q), "Round-trip must be lossless"
print(f"Round-trip verified: {q_restored.shape} matches original {q.shape}")
print(f"Max difference: {(q_restored - q).abs().max().item():.2e}  (should be 0.0)")

# Also verify the (B,S,N,D) -> (B,S,dim) round-trip
q_restored2 = rearrange(q_bsnd, "b s n d -> b s (n d)")     # [B, S, dim]
assert torch.allclose(q_restored2, q), "BSND round-trip must also be lossless"
print(f"BSND round-trip also verified: max diff = {(q_restored2 - q).abs().max().item():.2e}")

## Summary

### Key Takeaways
- **QKV projections in Wan2.1 are dim-to-dim**: `nn.Linear(dim, dim)` for Q, K, and V — no dimension expansion. Head splitting is a post-projection reshape, not a per-head projection.
- **Two head layout conventions coexist**: `(B, S, N, D)` (sequence-first) for flash_attn_2/3 backends and `(B, N, S, D)` (head-first) for PyTorch SDPA and SageAttn. The `flash_attention` function selects between them at runtime based on what is installed.
- **Head splitting is lossless**: `rearrange` is a pure reshape — merging heads back produces bit-identical output to the pre-split tensor. No information is lost or transformed.
- **head_dim drives RoPE**: The value `head_dim = dim // num_heads = 128` (for dim=1536, num_heads=12) determines the three frequency band sizes used by 3D RoPE (covered in NB-03).

### Source References
| Symbol | Location |
|--------|----------|
| SelfAttention (QKV projections) | `diffsynth/models/wan_video_dit.py`, line 125 |
| flash_attention (dual layout) | `diffsynth/models/wan_video_dit.py`, line 28 |

## Exercises

### Exercise 1 — Change head count
Change `num_heads` from 12 to 8 in Cell 4. Verify that `head_dim` becomes `1536 // 8 = 192` and all shapes update correctly. What happens to the total parameter count of each projection? (Hint: the projection size stays `dim x dim` — head count does not affect it.)

### Exercise 2 — Verify round-trip from (B,S,N,D)
After splitting to `(B,S,N,D)` in Cell 6, convert back to `(B,S,dim)` using `rearrange(q_bsnd, "b s n d -> b s (n d)")`. Verify it matches the original `q` tensor exactly using `torch.allclose`. Compare with the `(B,N,S,D)` round-trip already verified in Cell 8.

### Exercise 3 — Invalid head count
Try `dim=1536, num_heads=7` (1536 is not divisible by 7). What error does einops raise when you call `rearrange(q, "b s (n d) -> b s n d", n=7)`? Why is the constraint `dim % num_heads == 0` essential for head splitting?